# Track3 Mission Workbench (Dual Mode: Live/Simulation)

이 노트북은 Track3 미션 흐름(킥오프 → 통합응답 → fallback → 평가)을 **Dual Mode**로 실행합니다.

## 실행 모드
- `simulation` (기본): 로컬 생성 데이터로 재현 실행
- `live`: 실운영 커넥터 호출
  - **정형(Structured) 값은 FabricIQ만 사용**
  - **비정형(Unstructured) 근거는 WorkIQ만 사용**

## live 모드 필수 환경변수
- `TRACK3_EXECUTION_MODE=live`
- `FABRICIQ_ENDPOINT` (POST JSON 엔드포인트)
- `WORKIQ_ENDPOINT` (POST JSON 엔드포인트)

## live 모드 선택 환경변수
- `FABRICIQ_API_KEY`, `FABRICIQ_BEARER_TOKEN`
- `WORKIQ_API_KEY`, `WORKIQ_BEARER_TOKEN`
- `TRACK3_STRICT_EVAL=true|false` (기본: simulation만 strict)
- `FABRICIQ_TIMEOUT_SEC`, `WORKIQ_TIMEOUT_SEC` (기본 30초)

두 endpoint는 원시 제품 URL이 아니라 `scenarioId`, `question`, `semanticKeys`를 받는 POST JSON adapter입니다.
FabricIQ는 `structuredMetrics/highlights/sourceTrace`, WorkIQ는 `evidenceLinks/sourceCoverage/sourceTrace`를 반환하며,
인증·ACL·schema 오류를 성공 형태의 빈 응답으로 숨기지 않습니다.

> 목적: 실운영 경로와 로컬 재현 경로를 같은 노트북에서 검증하되, 데이터 소스 책임(정형=FabricIQ, 비정형=WorkIQ)을 명확히 분리합니다.

In [ ]:
from __future__ import annotations

import json
import os
import subprocess
import sys
import time
from datetime import datetime, timezone
from pathlib import Path
from urllib import error as urlerror
from urllib import request as urlrequest

ROOT = Path.cwd()
if (ROOT / 'track3' / 'data').is_dir():
    TRACK3_ROOT = ROOT / 'track3' / 'data'
elif ROOT.name == 'data' and ROOT.parent.name == 'track3':
    TRACK3_ROOT = ROOT
else:
    raise RuntimeError('track3/data 폴더를 찾을 수 없습니다. 저장소 루트 또는 track3/data 폴더에서 실행하세요.')

GENERATED_DIR = TRACK3_ROOT / 'generated'
RESPONSES_DIR = GENERATED_DIR / 'responses'
REPORTS_DIR = GENERATED_DIR / 'reports'

print('TRACK3_ROOT =', TRACK3_ROOT)

In [ ]:
EXECUTION_MODE = os.environ.get('TRACK3_EXECUTION_MODE', 'simulation').strip().lower()
if EXECUTION_MODE not in {'simulation', 'live'}:
    raise RuntimeError(f"지원하지 않는 TRACK3_EXECUTION_MODE: {EXECUTION_MODE}")

FABRICIQ_ENDPOINT = os.environ.get('FABRICIQ_ENDPOINT', '').strip()
FABRICIQ_API_KEY = os.environ.get('FABRICIQ_API_KEY', '').strip()
FABRICIQ_BEARER_TOKEN = os.environ.get('FABRICIQ_BEARER_TOKEN', '').strip()
FABRICIQ_TIMEOUT_SEC = int(os.environ.get('FABRICIQ_TIMEOUT_SEC', '30'))

WORKIQ_ENDPOINT = os.environ.get('WORKIQ_ENDPOINT', '').strip()
WORKIQ_API_KEY = os.environ.get('WORKIQ_API_KEY', '').strip()
WORKIQ_BEARER_TOKEN = os.environ.get('WORKIQ_BEARER_TOKEN', '').strip()
WORKIQ_TIMEOUT_SEC = int(os.environ.get('WORKIQ_TIMEOUT_SEC', '30'))

STRICT_EVAL = os.environ.get('TRACK3_STRICT_EVAL', 'true' if EXECUTION_MODE == 'simulation' else 'false').strip().lower() in {'1', 'true', 'yes', 'y'}

print('EXECUTION_MODE =', EXECUTION_MODE)
print('STRICT_EVAL =', STRICT_EVAL)
if EXECUTION_MODE == 'live':
    print('FABRICIQ_ENDPOINT =', FABRICIQ_ENDPOINT or '(미설정)')
    print('WORKIQ_ENDPOINT   =', WORKIQ_ENDPOINT or '(미설정)')


def run_cmd(args: list[str]) -> None:
    subprocess.run(args, check=True)


def clean_response_artifacts() -> None:
    RESPONSES_DIR.mkdir(parents=True, exist_ok=True)
    for path in RESPONSES_DIR.glob('*.json'):
        path.unlink()


def ensure_scenario_catalog() -> list[dict]:
    run_cmd([sys.executable, str(TRACK3_ROOT / 'generate_track3_samples.py'), '--pretty'])
    scenarios_path = GENERATED_DIR / 'scenarios.json'
    payload = json.loads(scenarios_path.read_text(encoding='utf-8'))
    return payload['scenarios']


def build_headers(api_key: str, bearer_token: str) -> dict[str, str]:
    headers = {'Content-Type': 'application/json; charset=utf-8'}
    if api_key:
        headers['x-api-key'] = api_key
    if bearer_token:
        headers['Authorization'] = f'Bearer {bearer_token}'
    return headers


def post_json(url: str, payload: dict, headers: dict[str, str], timeout_sec: int) -> dict:
    body = json.dumps(payload, ensure_ascii=False).encode('utf-8')
    retry_delays = [5, 10, 20]
    for attempt in range(len(retry_delays) + 1):
        req = urlrequest.Request(url=url, data=body, headers=headers, method='POST')
        try:
            with urlrequest.urlopen(req, timeout=timeout_sec) as resp:
                raw = resp.read().decode('utf-8')
            if not raw.strip():
                raise RuntimeError(f'Empty JSON response calling {url}')
            parsed = json.loads(raw)
            if not isinstance(parsed, dict):
                raise RuntimeError(f'Expected JSON object calling {url}')
            return parsed
        except urlerror.HTTPError as exc:
            detail = exc.read().decode('utf-8', errors='ignore')
            is_transient = exc.code == 429 or exc.code >= 500
            if not is_transient or attempt == len(retry_delays):
                raise RuntimeError(f'HTTPError {exc.code} calling {url}: {detail}') from exc
        except urlerror.URLError as exc:
            if attempt == len(retry_delays):
                raise RuntimeError(f'URLError calling {url}: {exc}') from exc
        delay = retry_delays[attempt]
        print(f'transient error calling {url}; retry {attempt + 1}/3 in {delay}s')
        time.sleep(delay)
    raise RuntimeError(f'Unreachable retry state calling {url}')


def fetch_fabriciq_structured(scenario: dict) -> dict:
    request_payload = {
        'scenarioId': scenario['id'],
        'question': scenario['question'],
        'goal': scenario.get('goal', ''),
        'semanticKeys': scenario.get('semanticKeys', []),
        'source': 'Track3_Mission_Workbench',
    }
    response_payload = post_json(
        FABRICIQ_ENDPOINT,
        request_payload,
        build_headers(FABRICIQ_API_KEY, FABRICIQ_BEARER_TOKEN),
        FABRICIQ_TIMEOUT_SEC,
    )
    highlights = response_payload.get('highlights') or response_payload.get('keyFindings') or []
    if isinstance(highlights, str):
        highlights = [highlights]
    structured_metrics = response_payload.get('structuredMetrics')
    if not isinstance(structured_metrics, dict) or not structured_metrics:
        raise RuntimeError(f"FabricIQ adapter response has no structuredMetrics for {scenario['id']}")
    if not highlights:
        raise RuntimeError(f"FabricIQ adapter response has no highlights for {scenario['id']}")
    normalized = dict(response_payload)
    normalized['highlights'] = list(highlights)
    return normalized


def normalize_evidence_items(raw_items: list[dict]) -> list[dict]:
    normalized = []
    for item in raw_items:
        normalized.append(
            {
                'source': item.get('source') or item.get('sourceType') or '-',
                'title': item.get('title') or item.get('name') or '-',
                'businessDate': item.get('businessDate') or item.get('date') or '-',
                'location': item.get('location') or item.get('url') or '-',
                'target': item.get('target') or item.get('id') or '-',
            }
        )
    return normalized


def fetch_workiq_evidence(scenario: dict) -> dict:
    request_payload = {
        'scenarioId': scenario['id'],
        'question': scenario['question'],
        'keywords': scenario.get('keywords', []),
        'semanticKeys': scenario.get('semanticKeys', []),
        'source': 'Track3_Mission_Workbench',
    }
    response_payload = post_json(
        WORKIQ_ENDPOINT,
        request_payload,
        build_headers(WORKIQ_API_KEY, WORKIQ_BEARER_TOKEN),
        WORKIQ_TIMEOUT_SEC,
    )
    raw_evidence = response_payload.get('evidence') or response_payload.get('evidenceLinks') or response_payload.get('results') or []
    if not isinstance(raw_evidence, list) or not raw_evidence:
        raise RuntimeError(f"WorkIQ adapter response has no evidence links for {scenario['id']}")
    normalized = dict(response_payload)
    normalized['evidence'] = normalize_evidence_items(raw_evidence)
    return normalized


def call_live_source(fetcher, scenario: dict, source_name: str) -> tuple[bool, dict | None, str | None]:
    try:
        return True, fetcher(scenario), None
    except RuntimeError as exc:
        error_message = f'{source_name} failed for {scenario["id"]}: {exc}'
        print(error_message)
        return False, None, error_message


In [ ]:
def compose_response(scenario: dict, tool_a_ok: bool, tool_b_ok: bool, tool_a_payload: dict | None, tool_b_payload: dict | None) -> dict:
    warnings: list[str] = []
    key_findings = list((tool_a_payload or {}).get('highlights', [])) if tool_a_ok else []
    evidence_links = list((tool_b_payload or {}).get('evidence', []))[:5] if tool_b_ok else []
    source_trace = []
    if tool_a_ok:
        source_trace.append({'iq': 'FabricIQ', 'role': 'structured', 'origin': f'{EXECUTION_MODE}-adapter' if EXECUTION_MODE == 'live' else 'track1-csv-simulation', 'semanticKeys': scenario.get('semanticKeys', [])})
    if tool_b_ok:
        source_trace.append({'iq': 'WorkIQ', 'role': 'unstructured', 'origin': f'{EXECUTION_MODE}-adapter' if EXECUTION_MODE == 'live' else 'track2-manifest-simulation', 'semanticKeys': scenario.get('semanticKeys', [])})

    if not tool_a_ok and not tool_b_ok:
        overall_status = 'blocked'
        warnings.append('Tool A/B 모두 실패: 답변 생성을 중단하고 차단 원인 및 복구 조치만 반환합니다.')
        key_findings = ['정형·비정형 도구가 모두 실패해 분석을 지속할 수 없습니다.']
        actions = [
            '권한/토큰 상태를 먼저 복구합니다.',
            '인덱스 범위와 커넥터 상태를 재점검합니다.',
            '복구 후 표준 질문 Q1으로 재시도합니다.',
        ]
        evidence_links = []
    elif not tool_a_ok:
        overall_status = 'partial'
        warnings.append('정형 수치 미검증')
        actions = [
            'Tool A(FabricIQ) 인증 또는 SQL endpoint 연결을 복구합니다.',
            '복구 후 동일 질문으로 정형 지표를 재수집합니다.',
        ]
    elif not tool_b_ok:
        overall_status = 'partial'
        warnings.append('업무 문서 근거 없음')
        actions = [
            'Tool B(WorkIQ) 권한/인덱스 최신성을 확인합니다.',
            '복구 후 동일 질문으로 근거 링크를 재수집합니다.',
        ]
    else:
        overall_status = 'pass'
        actions = [
            '근거 링크 접근 권한(ACL) 유효성을 교차 확인합니다.',
            '응답 품질 점수(정확도/근거성/환각률)를 기록합니다.',
        ]

    return {
        'question': scenario['question'],
        'overallStatus': overall_status,
        'summary': f"{scenario['id']} 실행 결과: {overall_status}",
        'keyFindings': key_findings,
        'warnings': warnings,
        'recommendedActions': actions,
        'evidenceLinks': evidence_links,
        'sourceTrace': source_trace,
        'qualityChecks': {
            'hasStructuredMetric': tool_a_ok and bool(key_findings or (tool_a_payload or {}).get('structuredMetrics')),
            'hasEvidenceLink': len(evidence_links) > 0,
            'hasBothSources': tool_a_ok and tool_b_ok and bool(key_findings or (tool_a_payload or {}).get('structuredMetrics')) and bool(evidence_links),
        },
    }


def build_result_payload(scenario: dict, mode: str, tool_a_ok: bool, tool_b_ok: bool, tool_a_payload: dict | None, tool_b_payload: dict | None, tool_errors: dict[str, str] | None = None) -> dict:
    retry_policy = {'maxRetries': 3, 'retryDelaysSec': [5, 10, 20]}
    tool_errors = tool_errors or {}
    response = compose_response(scenario, tool_a_ok, tool_b_ok, tool_a_payload, tool_b_payload)
    return {
        'runContext': {
            'scenarioId': scenario['id'],
            'executionMode': EXECUTION_MODE,
            'mode': mode,
            'runAt': datetime.now(timezone.utc).isoformat(),
            'retryPolicy': retry_policy,
            'release': {
                'pipelineVersion': 'track3-notebook-dualmode-v1',
                'promptVersion': 'track3-prompt-v1',
                'modelVersion': 'foundry-responses-v1',
                'toolsetVersion': 'fabriciq-live+workiq-live' if EXECUTION_MODE == 'live' else 'fabriciq-sim+workiq-sim',
            },
        },
        'toolStatus': {
            'toolA': {
                'status': 'ok' if tool_a_ok else 'fail',
                'attempts': 1 if tool_a_ok else 4,
                'logs': [{'attempt': 1 if tool_a_ok else 4, 'status': 'ok' if tool_a_ok else 'fail', **({'error': tool_errors.get('toolA', 'unavailable')} if not tool_a_ok else {})}],
            },
            'toolB': {
                'status': 'ok' if tool_b_ok else 'fail',
                'attempts': 1 if tool_b_ok else 4,
                'logs': [{'attempt': 1 if tool_b_ok else 4, 'status': 'ok' if tool_b_ok else 'fail', **({'error': tool_errors.get('toolB', 'unavailable')} if not tool_b_ok else {})}],
            },
        },
        'response': response,
    }


def save_response_file(payload: dict) -> Path:
    scenario_id = payload['runContext']['scenarioId']
    mode = payload['runContext']['mode']
    out = RESPONSES_DIR / f'{scenario_id}__{mode}.json'
    out.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding='utf-8')
    return out


In [ ]:
scenarios = ensure_scenario_catalog()
for scenario in scenarios:
    print(f"{scenario['id']}: {scenario['question']}")

clean_response_artifacts()

if EXECUTION_MODE == 'simulation':
    run_cmd([sys.executable, str(TRACK3_ROOT / 'run_track3_simulation.py'), '--all', '--mode', 'normal', '--pretty'])
    print('simulation(normal) 실행 완료')
else:
    if not FABRICIQ_ENDPOINT or not WORKIQ_ENDPOINT:
        raise RuntimeError('live 모드에서는 FABRICIQ_ENDPOINT와 WORKIQ_ENDPOINT가 모두 필요합니다.')

    live_cache: dict[str, dict] = {}
    for scenario in scenarios:
        tool_a_ok, structured, tool_a_error = call_live_source(fetch_fabriciq_structured, scenario, 'FabricIQ')
        tool_b_ok, evidence, tool_b_error = call_live_source(fetch_workiq_evidence, scenario, 'WorkIQ')
        payload = build_result_payload(
            scenario=scenario,
            mode='normal',
            tool_a_ok=tool_a_ok,
            tool_b_ok=tool_b_ok,
            tool_a_payload=structured,
            tool_b_payload=evidence,
            tool_errors={k: v for k, v in {'toolA': tool_a_error, 'toolB': tool_b_error}.items() if v},
        )
        out = save_response_file(payload)
        live_cache[scenario['id']] = {
            'scenario': scenario,
            'structured': structured,
            'evidence': evidence,
            'toolAOk': tool_a_ok,
            'toolBOk': tool_b_ok,
            'toolAError': tool_a_error,
            'toolBError': tool_b_error,
            'out': out,
        }
        print('live(normal) wrote:', out)

    LIVE_CACHE = live_cache
    print('live(normal) 실행 완료')

In [ ]:
if EXECUTION_MODE == 'simulation':
    for mode in ['tool-a-down', 'tool-b-down', 'both-down']:
        run_cmd([
            sys.executable,
            str(TRACK3_ROOT / 'run_track3_simulation.py'),
            '--scenario-id',
            'Q1',
            '--mode',
            mode,
            '--pretty',
        ])
    print('simulation(fallback) 실행 완료')
else:
    q1_ctx = LIVE_CACHE.get('Q1')
    if not q1_ctx:
        raise RuntimeError('live 모드 fallback 생성을 위해 Q1 normal 응답이 필요합니다.')

    q1 = q1_ctx['scenario']
    structured = q1_ctx['structured']
    evidence = q1_ctx['evidence']

    tool_a_down = build_result_payload(q1, 'tool-a-down', False, q1_ctx['toolBOk'], None, evidence, {'toolA': 'simulated tool-a-down', **({'toolB': q1_ctx['toolBError']} if q1_ctx['toolBError'] else {})})
    tool_b_down = build_result_payload(q1, 'tool-b-down', q1_ctx['toolAOk'], False, structured, None, {'toolB': 'simulated tool-b-down', **({'toolA': q1_ctx['toolAError']} if q1_ctx['toolAError'] else {})})
    both_down = build_result_payload(q1, 'both-down', False, False, None, None)

    print('live(fallback) wrote:', save_response_file(tool_a_down))
    print('live(fallback) wrote:', save_response_file(tool_b_down))
    print('live(fallback) wrote:', save_response_file(both_down))
    print('live(fallback) 실행 완료')

In [ ]:
eval_cmd = [sys.executable, str(TRACK3_ROOT / 'evaluate_track3_outputs.py'), '--pretty']
if STRICT_EVAL:
    eval_cmd.append('--strict')
run_cmd(eval_cmd)

report_path = REPORTS_DIR / 'evaluation_report.json'
markdown_report_path = REPORTS_DIR / 'evaluation_report.md'
report = json.loads(report_path.read_text(encoding='utf-8'))
print('평가 요약:', {k: report[k] for k in ['total', 'passed', 'failed']})
print('JSON 리포트:', report_path)
print('Markdown 리포트:', markdown_report_path)

# 아래 후속 셀과의 호환성 유지
responses_dir = RESPONSES_DIR

In [ ]:
for path in sorted(responses_dir.glob('*.json')):
    payload = json.loads(path.read_text(encoding='utf-8'))
    mode = payload['runContext']['mode']
    response = payload['response']
    print(f"{path.name}: mode={mode}, status={response['overallStatus']}, warnings={response['warnings']}")

## 질문에 대한 실제 답변 (Markdown)

`generated/responses/*.json`에는 각 시나리오 질문에 대한 실제 답변(`response` 객체: `summary`, `keyFindings`, `warnings`, `recommendedActions`, `evidenceLinks`)이 들어 있습니다. 아래 셀은 이를 사람이 읽기 쉬운 Markdown으로 변환해 `generated/reports/track3_answers.md`에 저장하고 노트북에 바로 렌더링합니다.

In [8]:
def render_answer_markdown(payload: dict) -> str:
    """Render a single Track3 response JSON payload as a Markdown section."""
    run_context = payload.get("runContext", {})
    response = payload.get("response", {})
    scenario_id = run_context.get("scenarioId", "?")
    mode = run_context.get("mode", "?")

    lines = [f"## {scenario_id} (mode={mode})", ""]
    lines.append(f"**질문:** {response.get('question', '-')}")
    lines.append("")
    lines.append(f"**상태:** {response.get('overallStatus', '-')}")
    lines.append("")
    lines.append(f"**요약:** {response.get('summary', '-')}")
    lines.append("")

    lines.append("### 핵심 발견 (keyFindings)")
    findings = response.get("keyFindings") or []
    lines.extend(f"- {item}" for item in findings) if findings else lines.append("- (없음)")
    lines.append("")

    warnings = response.get("warnings") or []
    if warnings:
        lines.append("### 경고 (warnings)")
        lines.extend(f"- ⚠️ {item}" for item in warnings)
        lines.append("")

    actions = response.get("recommendedActions") or []
    if actions:
        lines.append("### 권장 조치 (recommendedActions)")
        lines.extend(f"- {item}" for item in actions)
        lines.append("")

    trace = response.get("sourceTrace") or []
    lines.append("### 3-IQ 소스 추적 (sourceTrace)")
    if trace:
        lines.append("| iq | role | origin | semanticKeys |")
        lines.append("| --- | --- | --- | --- |")
        for item in trace:
            lines.append(f"| {item.get('iq', '-')} | {item.get('role', '-')} | {item.get('origin', '-')} | {', '.join(item.get('semanticKeys', []))} |")
    else:
        lines.append("- (없음)")
    lines.append("")

    links = response.get("evidenceLinks") or []
    lines.append("### 근거 링크 (evidenceLinks)")
    if links:
        lines.append("| source | title | businessDate | reference |")
        lines.append("| --- | --- | --- | --- |")
        for link in links:
            reference = link.get('url') or link.get('location') or link.get('target') or '-'
            lines.append(f"| {link.get('source', '-')} | {link.get('title', '-')} | {link.get('businessDate', '-')} | {reference} |")
    else:
        lines.append("- (없음)")
    lines.append("")
    lines.append("---")
    lines.append("")
    return "\n".join(lines)


answers_path = TRACK3_ROOT / "generated" / "reports" / "track3_answers.md"
response_files = sorted(responses_dir.glob("*.json"))

sections = ["# Track3 질문별 실제 답변", ""]
for path in response_files:
    payload = json.loads(path.read_text(encoding="utf-8"))
    sections.append(render_answer_markdown(payload))

answers_path.parent.mkdir(parents=True, exist_ok=True)
answers_path.write_text("\n".join(sections), encoding="utf-8")
print("답변 Markdown 저장 완료:", answers_path)


답변 Markdown 저장 완료: /Users/hyungilkim/Documents/Data Platform Workshop/track3/data/generated/reports/track3_answers.md


In [9]:
from IPython.display import Markdown, display

display(Markdown(answers_path.read_text(encoding='utf-8')))

# Track3 질문별 실제 답변

## Q1 (mode=both-down)

**질문:** 결제 실패가 캠페인 전환율에 미치는 영향은 무엇인가?

**상태:** blocked

**요약:** Q1 실행 결과: blocked

### 핵심 발견 (keyFindings)
- 정형·비정형 도구가 모두 실패해 분석을 지속할 수 없습니다.

### 경고 (warnings)
- ⚠️ Tool A/B 모두 실패: 답변 생성을 중단하고 차단 원인 및 복구 조치만 반환합니다.

### 권장 조치 (recommendedActions)
- 권한/토큰 상태를 먼저 복구합니다.
- 인덱스 범위와 커넥터 상태를 재점검합니다.
- 복구 후 표준 질문 Q1으로 재시도합니다.

### 근거 링크 (evidenceLinks)
- (없음)

---

## Q1 (mode=normal)

**질문:** 결제 실패가 캠페인 전환율에 미치는 영향은 무엇인가?

**상태:** pass

**요약:** Q1 실행 결과: pass

### 핵심 발견 (keyFindings)
- 핵심 캠페인 4개를 비교했고 최고 전환율은 BackToSchool, 최저 전환율은 VIPRetention이다.
- payment_status가 Success/RetrySuccess가 아닌 주문은 결제 실패/미확정으로 분류했다.

### 권장 조치 (recommendedActions)
- 근거 링크 접근 권한(ACL) 유효성을 교차 확인합니다.
- 응답 품질 점수(정확도/근거성/환각률)를 기록합니다.

### 근거 링크 (evidenceLinks)
| source | title | businessDate |
| --- | --- | --- |
| Teams | SummerPush 중간 성과 해석 | 2026-05-20T15:00:00+09:00 |
| SharePoint | SummerPush 중간 성과 리포트 | 2026-05-20 |
| Outlook | [리더십] 5월 매출 급락 이슈 공유 | 2026-05-18T08:40:00+09:00 |
| OneDrive | 캠페인 주간 성과 리뷰 노트 | 2026-05-21 |
| SharePoint | SummerPush 캠페인 킥오프 기획서 | 2026-04-15 |

---

## Q1 (mode=tool-a-down)

**질문:** 결제 실패가 캠페인 전환율에 미치는 영향은 무엇인가?

**상태:** partial

**요약:** Q1 실행 결과: partial

### 핵심 발견 (keyFindings)
- (없음)

### 경고 (warnings)
- ⚠️ 정형 수치 미검증

### 권장 조치 (recommendedActions)
- Tool A(FabricIQ) 인증 또는 SQL endpoint 연결을 복구합니다.
- 복구 후 동일 질문으로 정형 지표를 재수집합니다.

### 근거 링크 (evidenceLinks)
| source | title | businessDate |
| --- | --- | --- |
| Teams | SummerPush 중간 성과 해석 | 2026-05-20T15:00:00+09:00 |
| SharePoint | SummerPush 중간 성과 리포트 | 2026-05-20 |
| Outlook | [리더십] 5월 매출 급락 이슈 공유 | 2026-05-18T08:40:00+09:00 |
| OneDrive | 캠페인 주간 성과 리뷰 노트 | 2026-05-21 |
| SharePoint | SummerPush 캠페인 킥오프 기획서 | 2026-04-15 |

---

## Q1 (mode=tool-b-down)

**질문:** 결제 실패가 캠페인 전환율에 미치는 영향은 무엇인가?

**상태:** partial

**요약:** Q1 실행 결과: partial

### 핵심 발견 (keyFindings)
- 핵심 캠페인 4개를 비교했고 최고 전환율은 BackToSchool, 최저 전환율은 VIPRetention이다.
- payment_status가 Success/RetrySuccess가 아닌 주문은 결제 실패/미확정으로 분류했다.

### 경고 (warnings)
- ⚠️ 업무 문서 근거 없음

### 권장 조치 (recommendedActions)
- Tool B(WorkIQ) 권한/인덱스 최신성을 확인합니다.
- 복구 후 동일 질문으로 근거 링크를 재수집합니다.

### 근거 링크 (evidenceLinks)
- (없음)

---

## Q2 (mode=normal)

**질문:** 배송 지연은 반품률과 고객 불만 티켓에 어떤 영향을 미치는가?

**상태:** pass

**요약:** Q2 실행 결과: pass

### 핵심 발견 (keyFindings)
- 배송 지연 주문 669건 중 반품 발생 비율은 49.78%이다.
- 배송 지연 주문의 COMPLAINT 티켓 비율은 16.14%이다.

### 권장 조치 (recommendedActions)
- 근거 링크 접근 권한(ACL) 유효성을 교차 확인합니다.
- 응답 품질 점수(정확도/근거성/환각률)를 기록합니다.

### 근거 링크 (evidenceLinks)
| source | title | businessDate |
| --- | --- | --- |
| Teams | 주문 취소율과 배송 지연 상관 점검 | 2026-05-24T14:00:00+09:00 |
| SharePoint | 배송 지연 원인 분석 및 고객 영향 | 2026-05-23 |
| Outlook | 반품 사유 월간 요약 - 채널 및 고객등급 검토 | 2026-05-25T16:30:00+09:00 |
| OneDrive | 반품 VOC 분류 워크숍 노트 | 2026-05-25 |
| Outlook | RE: [긴급] 핵심 상품 재고 부족 및 캠페인 노출 조정 요청 | 2026-05-17T08:50:00+09:00 |

---

## Q2 (mode=tool-a-transient)

**질문:** 배송 지연은 반품률과 고객 불만 티켓에 어떤 영향을 미치는가?

**상태:** pass

**요약:** Q2 실행 결과: pass

### 핵심 발견 (keyFindings)
- 배송 지연 주문 669건 중 반품 발생 비율은 49.78%이다.
- 배송 지연 주문의 COMPLAINT 티켓 비율은 16.14%이다.

### 권장 조치 (recommendedActions)
- 근거 링크 접근 권한(ACL) 유효성을 교차 확인합니다.
- 응답 품질 점수(정확도/근거성/환각률)를 기록합니다.

### 근거 링크 (evidenceLinks)
| source | title | businessDate |
| --- | --- | --- |
| Teams | 주문 취소율과 배송 지연 상관 점검 | 2026-05-24T14:00:00+09:00 |
| SharePoint | 배송 지연 원인 분석 및 고객 영향 | 2026-05-23 |
| Outlook | 반품 사유 월간 요약 - 채널 및 고객등급 검토 | 2026-05-25T16:30:00+09:00 |
| OneDrive | 반품 VOC 분류 워크숍 노트 | 2026-05-25 |
| Outlook | RE: [긴급] 핵심 상품 재고 부족 및 캠페인 노출 조정 요청 | 2026-05-17T08:50:00+09:00 |

---

## Q3 (mode=normal)

**질문:** Q3 핵심 상품 3종의 매출/반품 신호를 어떻게 해석할 것인가?

**상태:** pass

**요약:** Q3 실행 결과: pass

### 핵심 발견 (keyFindings)
- Q3 핵심 상품 3종(AeroPhone X, SmartWatch Pro, UltraBook 15)을 동일 기준으로 비교했다.
- 주문 수, 매출, 반품률을 함께 보고 대응 우선순위를 선정한다.

### 권장 조치 (recommendedActions)
- 근거 링크 접근 권한(ACL) 유효성을 교차 확인합니다.
- 응답 품질 점수(정확도/근거성/환각률)를 기록합니다.

### 근거 링크 (evidenceLinks)
| source | title | businessDate |
| --- | --- | --- |
| SharePoint | Q3 리더십 운영 리스크 브리핑 | 2026-07-11 |
| Outlook | RE: BackToSchool 캠페인 조건부 승인 요청 | 2026-07-11T09:30:00+09:00 |
| OneDrive | 재고·물류·CS 합동 회의록 | 2026-05-17 |
| Teams | 핵심 상품 품절 임박 공동 대응 | 2026-05-16T09:00:00+09:00 |
| Outlook | [긴급] 핵심 상품 재고 부족 및 캠페인 노출 조정 요청 | 2026-05-16T09:15:00+09:00 |

---

## Q3 (mode=tool-b-transient)

**질문:** Q3 핵심 상품 3종의 매출/반품 신호를 어떻게 해석할 것인가?

**상태:** pass

**요약:** Q3 실행 결과: pass

### 핵심 발견 (keyFindings)
- Q3 핵심 상품 3종(AeroPhone X, SmartWatch Pro, UltraBook 15)을 동일 기준으로 비교했다.
- 주문 수, 매출, 반품률을 함께 보고 대응 우선순위를 선정한다.

### 권장 조치 (recommendedActions)
- 근거 링크 접근 권한(ACL) 유효성을 교차 확인합니다.
- 응답 품질 점수(정확도/근거성/환각률)를 기록합니다.

### 근거 링크 (evidenceLinks)
| source | title | businessDate |
| --- | --- | --- |
| SharePoint | Q3 리더십 운영 리스크 브리핑 | 2026-07-11 |
| Outlook | RE: BackToSchool 캠페인 조건부 승인 요청 | 2026-07-11T09:30:00+09:00 |
| OneDrive | 재고·물류·CS 합동 회의록 | 2026-05-17 |
| Teams | 핵심 상품 품절 임박 공동 대응 | 2026-05-16T09:00:00+09:00 |
| Outlook | [긴급] 핵심 상품 재고 부족 및 캠페인 노출 조정 요청 | 2026-05-16T09:15:00+09:00 |

---


## 평가 리포트 (Markdown)

`evaluate_track3_outputs.py`는 JSON 리포트를 만든 직후 동일 내용을 Markdown 표로도 저장합니다. 아래 셀에서 바로 렌더링해 확인합니다.

In [10]:
from IPython.display import Markdown, display

markdown_report_path = TRACK3_ROOT / 'generated' / 'reports' / 'evaluation_report.md'
display(Markdown(markdown_report_path.read_text(encoding='utf-8')))

# Track3 Evaluation Report

- responsesDir: `/Users/hyungilkim/Documents/Data Platform Workshop/track3/data/generated/responses`
- total: 8
- passed: 8
- failed: 0

## Results

| Scenario | Mode | Result | Reasons |
| --- | --- | --- | --- |
| Q1 | both-down | ✅ PASS | - |
| Q1 | normal | ✅ PASS | - |
| Q1 | tool-a-down | ✅ PASS | - |
| Q1 | tool-b-down | ✅ PASS | - |
| Q2 | normal | ✅ PASS | - |
| Q2 | tool-a-transient | ✅ PASS | - |
| Q3 | normal | ✅ PASS | - |
| Q3 | tool-b-transient | ✅ PASS | - |


## 임원용 리더십 브리핑 초안

[track3/WORKBOOK.md](../WORKBOOK.md)의 **미션 4**에서는 참가자가 FoundryIQ 에이전트에게 리더십 브리핑 생성을 요청해
`[TRACK3_RESPONSE]` 형식의 응답을 만들고, Q1~Q3를 묶은 최종 브리핑 1건을 제출합니다.

여기서는 실제 LLM 생성 없이 Q1~Q3 정상(normal) 응답을 규칙 기반으로 결합해 임원이 바로 읽을 수 있는
1페이지 브리핑 초안을 자동으로 만들고, `generated/reports/leadership_briefing.md`에 저장합니다.


In [11]:
def render_track3_response_block(payload: dict) -> str:
    """WORKBOOK.md의 [TRACK3_RESPONSE] 고정 출력 형식으로 단일 시나리오 응답을 렌더링합니다."""
    response = payload.get("response", {})
    metrics = response.get("keyFindings") or ["-"]
    links = response.get("evidenceLinks") or []
    link_summary = "; ".join(f"{l.get('source', '-')}:{l.get('title', '-')}" for l in links) or "-"
    source_trace = response.get("sourceTrace") or []
    trace_summary = "; ".join(f"{item.get('iq', '-')}:{item.get('role', '-')}" for item in source_trace) or "-"
    actions = response.get("recommendedActions") or ["-"]
    warnings = response.get("warnings") or []

    lines = [
        "[TRACK3_RESPONSE]",
        f"question={response.get('question', '-')}",
        f"summary={response.get('summary', '-')}",
        f"structuredMetrics={' / '.join(metrics)}",
        f"evidenceLinks={link_summary}",
        f"sourceTrace={trace_summary}",
        f"actions={' / '.join(actions)}",
        f"warnings={'; '.join(warnings) if warnings else '없음'}",
        "[/TRACK3_RESPONSE]",
    ]
    return "\n".join(lines)


def compose_leadership_briefing(scenario_payloads: dict) -> str:
    """Q1~Q3 정상 응답을 결합해 임원용 리더십 브리핑 초안(Markdown)을 생성합니다."""
    from datetime import datetime, timezone

    generated_at = datetime.now(timezone.utc).isoformat()
    lines = []
    lines.append("# Track3 리더십 브리핑 (자동 초안)")
    lines.append("")
    lines.append(f"- 생성일시(UTC): {generated_at}")
    lines.append("- 대상 시나리오: " + ", ".join(sorted(scenario_payloads.keys())))
    lines.append("")

    lines.append("## 한눈에 보기 (Executive Summary)")
    for scenario_id in sorted(scenario_payloads.keys()):
        response = scenario_payloads[scenario_id]["response"]
        first_finding = (response.get("keyFindings") or ["-"])[0]
        lines.append(f"- **{scenario_id}** — {response.get('question', '-')}: {first_finding}")
    lines.append("")

    lines.append("## 핵심 수치 근거 (Structured Metrics)")
    for scenario_id in sorted(scenario_payloads.keys()):
        response = scenario_payloads[scenario_id]["response"]
        lines.append(f"### {scenario_id}")
        for finding in response.get("keyFindings") or ["-"]:
            lines.append(f"- {finding}")
    lines.append("")

    lines.append("## 문서 근거 (Evidence Links, 시나리오별 상위 3건)")
    lines.append("| Scenario | source | title | businessDate |")
    lines.append("| --- | --- | --- | --- |")
    for scenario_id in sorted(scenario_payloads.keys()):
        response = scenario_payloads[scenario_id]["response"]
        for link in (response.get("evidenceLinks") or [])[:3]:
            lines.append(f"| {scenario_id} | {link.get('source', '-')} | {link.get('title', '-')} | {link.get('businessDate', '-')} |")
    lines.append("")

    lines.append("## 즉시 조치 제안 (Actions)")
    seen_actions = set()
    for scenario_id in sorted(scenario_payloads.keys()):
        response = scenario_payloads[scenario_id]["response"]
        for action in response.get("recommendedActions") or []:
            if action not in seen_actions:
                lines.append(f"- {action}")
                seen_actions.add(action)
    lines.append("")

    lines.append("## 주의사항 (Warnings)")
    any_warning = False
    for scenario_id in sorted(scenario_payloads.keys()):
        response = scenario_payloads[scenario_id]["response"]
        for warning in response.get("warnings") or []:
            lines.append(f"- **{scenario_id}**: ⚠️ {warning}")
            any_warning = True
    if not any_warning:
        lines.append("- 없음 (Q1~Q3 모두 정형 수치 + 근거 링크 정상 확보)")
    lines.append("")

    lines.append("## Appendix — 시나리오별 [TRACK3_RESPONSE] 제출 블록")
    lines.append("")
    lines.append("```text")
    for scenario_id in sorted(scenario_payloads.keys()):
        lines.append(render_track3_response_block(scenario_payloads[scenario_id]))
        lines.append("")
    lines.append("```")
    lines.append("")

    return "\n".join(lines)


normal_payloads = {}
for path in sorted(responses_dir.glob("*__normal.json")):
    payload = json.loads(path.read_text(encoding="utf-8"))
    normal_payloads[payload["runContext"]["scenarioId"]] = payload

briefing_path = TRACK3_ROOT / "generated" / "reports" / "leadership_briefing.md"
briefing_markdown = compose_leadership_briefing(normal_payloads)
briefing_path.parent.mkdir(parents=True, exist_ok=True)
briefing_path.write_text(briefing_markdown, encoding="utf-8")
print("리더십 브리핑 초안 저장 완료:", briefing_path)


리더십 브리핑 초안 저장 완료: /Users/hyungilkim/Documents/Data Platform Workshop/track3/data/generated/reports/leadership_briefing.md


In [12]:
from IPython.display import Markdown, display

display(Markdown(briefing_path.read_text(encoding='utf-8')))

# Track3 리더십 브리핑 (자동 초안)

- 생성일시(UTC): 2026-07-13T18:13:46.552610+00:00
- 대상 시나리오: Q1, Q2, Q3

## 한눈에 보기 (Executive Summary)
- **Q1** — 결제 실패가 캠페인 전환율에 미치는 영향은 무엇인가?: 핵심 캠페인 4개를 비교했고 최고 전환율은 BackToSchool, 최저 전환율은 VIPRetention이다.
- **Q2** — 배송 지연은 반품률과 고객 불만 티켓에 어떤 영향을 미치는가?: 배송 지연 주문 669건 중 반품 발생 비율은 49.78%이다.
- **Q3** — Q3 핵심 상품 3종의 매출/반품 신호를 어떻게 해석할 것인가?: Q3 핵심 상품 3종(AeroPhone X, SmartWatch Pro, UltraBook 15)을 동일 기준으로 비교했다.

## 핵심 수치 근거 (Structured Metrics)
### Q1
- 핵심 캠페인 4개를 비교했고 최고 전환율은 BackToSchool, 최저 전환율은 VIPRetention이다.
- payment_status가 Success/RetrySuccess가 아닌 주문은 결제 실패/미확정으로 분류했다.
### Q2
- 배송 지연 주문 669건 중 반품 발생 비율은 49.78%이다.
- 배송 지연 주문의 COMPLAINT 티켓 비율은 16.14%이다.
### Q3
- Q3 핵심 상품 3종(AeroPhone X, SmartWatch Pro, UltraBook 15)을 동일 기준으로 비교했다.
- 주문 수, 매출, 반품률을 함께 보고 대응 우선순위를 선정한다.

## 문서 근거 (Evidence Links, 시나리오별 상위 3건)
| Scenario | source | title | businessDate |
| --- | --- | --- | --- |
| Q1 | Teams | SummerPush 중간 성과 해석 | 2026-05-20T15:00:00+09:00 |
| Q1 | SharePoint | SummerPush 중간 성과 리포트 | 2026-05-20 |
| Q1 | Outlook | [리더십] 5월 매출 급락 이슈 공유 | 2026-05-18T08:40:00+09:00 |
| Q2 | Teams | 주문 취소율과 배송 지연 상관 점검 | 2026-05-24T14:00:00+09:00 |
| Q2 | SharePoint | 배송 지연 원인 분석 및 고객 영향 | 2026-05-23 |
| Q2 | Outlook | 반품 사유 월간 요약 - 채널 및 고객등급 검토 | 2026-05-25T16:30:00+09:00 |
| Q3 | SharePoint | Q3 리더십 운영 리스크 브리핑 | 2026-07-11 |
| Q3 | Outlook | RE: BackToSchool 캠페인 조건부 승인 요청 | 2026-07-11T09:30:00+09:00 |
| Q3 | OneDrive | 재고·물류·CS 합동 회의록 | 2026-05-17 |

## 즉시 조치 제안 (Actions)
- 근거 링크 접근 권한(ACL) 유효성을 교차 확인합니다.
- 응답 품질 점수(정확도/근거성/환각률)를 기록합니다.

## 주의사항 (Warnings)
- 없음 (Q1~Q3 모두 정형 수치 + 근거 링크 정상 확보)

## Appendix — 시나리오별 [TRACK3_RESPONSE] 제출 블록

```text
[TRACK3_RESPONSE]
question=결제 실패가 캠페인 전환율에 미치는 영향은 무엇인가?
summary=Q1 실행 결과: pass
structuredMetrics=핵심 캠페인 4개를 비교했고 최고 전환율은 BackToSchool, 최저 전환율은 VIPRetention이다. / payment_status가 Success/RetrySuccess가 아닌 주문은 결제 실패/미확정으로 분류했다.
evidenceLinks=Teams:SummerPush 중간 성과 해석; SharePoint:SummerPush 중간 성과 리포트; Outlook:[리더십] 5월 매출 급락 이슈 공유; OneDrive:캠페인 주간 성과 리뷰 노트; SharePoint:SummerPush 캠페인 킥오프 기획서
actions=근거 링크 접근 권한(ACL) 유효성을 교차 확인합니다. / 응답 품질 점수(정확도/근거성/환각률)를 기록합니다.
warnings=없음
[/TRACK3_RESPONSE]

[TRACK3_RESPONSE]
question=배송 지연은 반품률과 고객 불만 티켓에 어떤 영향을 미치는가?
summary=Q2 실행 결과: pass
structuredMetrics=배송 지연 주문 669건 중 반품 발생 비율은 49.78%이다. / 배송 지연 주문의 COMPLAINT 티켓 비율은 16.14%이다.
evidenceLinks=Teams:주문 취소율과 배송 지연 상관 점검; SharePoint:배송 지연 원인 분석 및 고객 영향; Outlook:반품 사유 월간 요약 - 채널 및 고객등급 검토; OneDrive:반품 VOC 분류 워크숍 노트; Outlook:RE: [긴급] 핵심 상품 재고 부족 및 캠페인 노출 조정 요청
actions=근거 링크 접근 권한(ACL) 유효성을 교차 확인합니다. / 응답 품질 점수(정확도/근거성/환각률)를 기록합니다.
warnings=없음
[/TRACK3_RESPONSE]

[TRACK3_RESPONSE]
question=Q3 핵심 상품 3종의 매출/반품 신호를 어떻게 해석할 것인가?
summary=Q3 실행 결과: pass
structuredMetrics=Q3 핵심 상품 3종(AeroPhone X, SmartWatch Pro, UltraBook 15)을 동일 기준으로 비교했다. / 주문 수, 매출, 반품률을 함께 보고 대응 우선순위를 선정한다.
evidenceLinks=SharePoint:Q3 리더십 운영 리스크 브리핑; Outlook:RE: BackToSchool 캠페인 조건부 승인 요청; OneDrive:재고·물류·CS 합동 회의록; Teams:핵심 상품 품절 임박 공동 대응; Outlook:[긴급] 핵심 상품 재고 부족 및 캠페인 노출 조정 요청
actions=근거 링크 접근 권한(ACL) 유효성을 교차 확인합니다. / 응답 품질 점수(정확도/근거성/환각률)를 기록합니다.
warnings=없음
[/TRACK3_RESPONSE]

```


## FoundryIQ 에이전트(LLM)로 최종 리더십 브리핑 문서 생성

위에서 만든 `leadership_briefing.md`는 **규칙 기반 초안**입니다. 이 섹션은 실제 **Azure AI Foundry Responses API**를
호출해, 그 초안을 [WORKBOOK.md](../WORKBOOK.md) 미션 2의 시스템 프롬프트 정책(정형 우선 + 근거 결합, 근거 없는
단정 금지, `핵심요약/수치근거/문서근거/조치안/주의사항` 출력 형식)에 맞춰 임원이 바로 읽을 수 있는 완성된 문장으로 다듬습니다.

### 사전 준비 (환경변수)

| 환경변수 | 설명 | 필수 여부 |
|---|---|---|
| `AZURE_AI_FOUNDRY_RESPONSES_ENDPOINT` | Responses API 엔드포인트 (예: `https://<resource>.services.ai.azure.com/openai/v1/responses`) | 필수 |
| `AZURE_AI_FOUNDRY_MODEL` | 호출할 모델/배포 이름 (예: `gpt-5.6`, `gpt-4o-mini`) | 필수 |
| `AZURE_AI_FOUNDRY_API_KEY` | 키 인증 사용 시 API Key (`api-key` 헤더) | 선택 |
| `AZURE_AI_FOUNDRY_BEARER_TOKEN` | Bearer 토큰 인증 사용 시 토큰 (`Authorization` 헤더) | 선택 |

> `AZURE_AI_FOUNDRY_API_KEY`와 `AZURE_AI_FOUNDRY_BEARER_TOKEN` 중 **하나 이상**이 있어야 호출할 수 있습니다.
> JWT 액세스 토큰을 API key 변수에 넣지 마세요. 두 값이 모두 있으면 Bearer token을 우선합니다.
> 값이 없으면 이 섹션은 자동으로 건너뛰고, 노트북 나머지 셀은 정상 동작합니다.


In [ ]:
import os
import sys

if str(TRACK3_ROOT) not in sys.path:
    sys.path.insert(0, str(TRACK3_ROOT))

from foundry_responses import FoundryResponsesConfig, generate_leadership_briefing

print('Foundry Responses API 공용 모듈 로드 완료')


In [ ]:
# Foundry Responses API 연결 정보 (환경변수에서 읽음, 코드에 하드코딩하지 않음)
foundry_config = FoundryResponsesConfig.from_env()
AZURE_AI_FOUNDRY_RESPONSES_ENDPOINT = foundry_config.endpoint
AZURE_AI_FOUNDRY_MODEL = foundry_config.model
AZURE_AI_FOUNDRY_API_KEY = foundry_config.api_key
AZURE_AI_FOUNDRY_BEARER_TOKEN = foundry_config.bearer_token

HAS_AUTH = bool(AZURE_AI_FOUNDRY_API_KEY or AZURE_AI_FOUNDRY_BEARER_TOKEN)
FOUNDRY_CONFIGURED = foundry_config.is_configured

if AZURE_AI_FOUNDRY_BEARER_TOKEN:
    auth_mode = "Bearer Token"
elif AZURE_AI_FOUNDRY_API_KEY:
    auth_mode = "API Key"
else:
    auth_mode = "(미설정)"

print("AZURE_AI_FOUNDRY_RESPONSES_ENDPOINT =", AZURE_AI_FOUNDRY_RESPONSES_ENDPOINT or "(미설정)")
print("AZURE_AI_FOUNDRY_MODEL              =", AZURE_AI_FOUNDRY_MODEL or "(미설정)")
print("AZURE_AI_FOUNDRY_API_KEY            =", "설정됨(마스킹)" if AZURE_AI_FOUNDRY_API_KEY else "(미설정)")
print("AZURE_AI_FOUNDRY_BEARER_TOKEN       =", "설정됨(마스킹)" if AZURE_AI_FOUNDRY_BEARER_TOKEN else "(미설정)")
print("인증 방식                            =", auth_mode)
print("FOUNDRY_CONFIGURED                  =", FOUNDRY_CONFIGURED)


In [ ]:
FOUNDRY_SYSTEM_PROMPT = """당신은 Track3 FoundryIQ 리더십 브리핑 작성 에이전트입니다.
다음 고정 정책을 반드시 지키세요:
- Tool A(FabricIQ) 정형 수치를 우선 반영하고 Tool B(WorkIQ) 문서 근거를 결합합니다.
- 근거 링크가 없는 문장은 단정하지 말고 경고를 명시합니다.
- 출력은 다음 5개 섹션 형식을 고정합니다: 핵심요약, 수치근거, 문서근거, 조치안, 주의사항.
- 한국어로 작성하고, 임원이 1분 내로 읽을 수 있도록 간결하게 작성합니다."""


def build_foundry_headers() -> dict[str, str]:
    headers = {
        "Content-Type": "application/json; charset=utf-8",
    }
    if AZURE_AI_FOUNDRY_BEARER_TOKEN:
        headers["Authorization"] = f"Bearer {AZURE_AI_FOUNDRY_BEARER_TOKEN}"
    elif AZURE_AI_FOUNDRY_API_KEY:
        headers["api-key"] = AZURE_AI_FOUNDRY_API_KEY
    return headers


def extract_responses_output_text(payload: dict) -> str:
    if isinstance(payload.get("output_text"), str) and payload["output_text"].strip():
        return payload["output_text"].strip()

    for message in payload.get("output", []) or []:
        for part in message.get("content", []) or []:
            text = part.get("text")
            if isinstance(text, str) and text.strip():
                return text.strip()

    return ""


def call_foundry_responses_api(*, system_prompt: str, user_prompt: str, model: str, max_output_tokens: int = 1400) -> dict:
    request_body = {
        "model": model,
        "input": [
            {
                "role": "system",
                "content": [
                    {"type": "input_text", "text": system_prompt}
                ],
            },
            {
                "role": "user",
                "content": [
                    {"type": "input_text", "text": user_prompt}
                ],
            },
        ],
        "max_output_tokens": max_output_tokens,
    }

    raw = post_json(
        AZURE_AI_FOUNDRY_RESPONSES_ENDPOINT,
        request_body,
        build_foundry_headers(),
        timeout_sec=60,
    )
    return raw


def generate_llm_leadership_briefing(draft_markdown: str) -> str:
    """규칙 기반 초안을 Foundry Responses API로 다듬어 최종 브리핑 문장을 생성합니다."""
    return generate_leadership_briefing(draft_markdown, config=foundry_config)


print("generate_llm_leadership_briefing (Responses API) 정의 완료")


In [ ]:
llm_briefing_path = TRACK3_ROOT / "generated" / "reports" / "leadership_briefing_llm.md"

if not FOUNDRY_CONFIGURED:
    print("[건너뜀] Foundry Responses API 환경변수가 설정되지 않아 LLM 기반 최종본 생성을 건너뜁니다.")
    print("필수: AZURE_AI_FOUNDRY_RESPONSES_ENDPOINT, AZURE_AI_FOUNDRY_MODEL, 인증(API_KEY 또는 BEARER_TOKEN)")
    print("규칙 기반 초안(generated/reports/leadership_briefing.md)을 그대로 제출용으로 사용할 수 있습니다.")
else:
    draft_markdown = briefing_path.read_text(encoding="utf-8")
    llm_output = generate_llm_leadership_briefing(draft_markdown)
    llm_briefing_path.parent.mkdir(parents=True, exist_ok=True)
    llm_briefing_path.write_text(llm_output, encoding="utf-8")
    print("Foundry Responses API 최종 브리핑 저장 완료:", llm_briefing_path)

    from IPython.display import Markdown, display
    display(Markdown(llm_output))
